In [1]:
!pip install requests biopython scikit-learn pandas -q

import requests
import time
import pandas as pd
import numpy as np

# Known human transcription factors (sequence-specific DNA-binding regulators)
tf_genes = [
    "TP53","MYC","JUN","FOS","GATA1","GATA2","GATA3","GATA4","SOX2","POU5F1",
    "KLF4","NANOG","STAT3","NFKB1","RELA","CREB1","E2F1","FOXP3","PAX6","HOXA1",
    "IRF1","SMAD3","TFAP2A","ETS1","RUNX1","CEBPA","MYOD1","SP1","TBX5","ATF4",
    "MAFA","PPARG","RXRA","ESR1","AR","VDR","THRA","HNF4A","FOXA2","PDX1",
    "NEUROD1","ASCL1","ZEB1","SNAI1","TWIST1","MEF2C","STAT1","IRF3","NFATC1","FOXO1"
]

# Known non-TF human proteins (enzymes, structural, signaling - not sequence-specific DNA binders)
non_tf_genes = [
    "ACTB","GAPDH","ALB","INS","HBB","HBA1","LDHA","PKM","ENO1","ALDOA",
    "TPI1","PGK1","CAT","SOD1","SOD2","GPX1","MB","TTN","COL1A1","FN1",
    "TUBB","ACTA1","MYH7","DES","VIM","KRT8","KRT18","CASP3","BAX","BCL2",
    "CYCS","ATP5F1A","NDUFA1","COX4I1","UQCRC1","PFKM","HK1","G6PD","PRKACA","MAPK1",
    "AKT1","PTEN","MTOR","RAF1","GSK3B","CDK2","CCNB1","PLK1","AURKA","RPS6"
]

print(f"TF genes: {len(tf_genes)}")
print(f"Non-TF genes: {len(non_tf_genes)}")
print("Setup done")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 9.5 MB/s eta 0:00:00a 0:00:01
TF genes: 50
Non-TF genes: 50
Setup done


In [2]:
def fetch_uniprot_sequence(gene_name, organism="9606"):
    """Query UniProt REST API for the reviewed (Swiss-Prot) entry matching this gene name."""
    url = "https://rest.uniprot.org/uniprotkb/search"
    params = {
        "query": f"gene:{gene_name} AND organism_id:{organism} AND reviewed:true",
        "fields": "sequence",
        "format": "json",
        "size": 1
    }
    try:
        r = requests.get(url, params=params, timeout=15)
        data = r.json()
        if data.get("results"):
            seq = data["results"][0]["sequence"]["value"]
            return seq
    except Exception as e:
        print(f"  {gene_name}: FAILED - {e}")
    return None

def fetch_all(gene_list, label):
    records = []
    for g in gene_list:
        seq = fetch_uniprot_sequence(g)
        if seq:
            records.append({"gene": g, "sequence": seq, "is_tf": label})
            print(f"{g}: {len(seq)} aa")
        else:
            print(f"{g}: NOT FOUND")
        time.sleep(0.3)
    return records

print("=== Fetching TF sequences ===")
tf_records = fetch_all(tf_genes, 1)

print("\n=== Fetching non-TF sequences ===")
non_tf_records = fetch_all(non_tf_genes, 0)

df = pd.DataFrame(tf_records + non_tf_records)
print(f"\nTotal sequences fetched: {len(df)}")
print(df.groupby("is_tf").size())

=== Fetching TF sequences ===
TP53: 393 aa
MYC: 454 aa
JUN: 331 aa
FOS: 380 aa
GATA1: 413 aa
GATA2: 480 aa
GATA3: 443 aa
GATA4: 442 aa
SOX2: 317 aa
POU5F1: 360 aa
KLF4: 513 aa
NANOG: 305 aa
STAT3: 770 aa
NFKB1: 968 aa
RELA: 551 aa
CREB1: 327 aa
E2F1: 437 aa
FOXP3: 431 aa
PAX6: 422 aa
HOXA1: 335 aa
IRF1: 325 aa
SMAD3: 425 aa
TFAP2A: 437 aa
ETS1: 441 aa
RUNX1: 453 aa
CEBPA: 358 aa
MYOD1: 320 aa
SP1: 785 aa
TBX5: 518 aa
ATF4: 351 aa
MAFA: 353 aa
PPARG: 505 aa
RXRA: 462 aa
ESR1: 595 aa
AR: 920 aa
VDR: 427 aa
THRA: 490 aa
HNF4A: 474 aa
FOXA2: 457 aa
PDX1: 501 aa
NEUROD1: 356 aa
ASCL1: 236 aa
ZEB1: 1124 aa
SNAI1: 264 aa
TWIST1: 202 aa
MEF2C: 473 aa
STAT1: 750 aa
IRF3: 427 aa
NFATC1: 943 aa
FOXO1: 655 aa

=== Fetching non-TF sequences ===
ACTB: 375 aa
GAPDH: 335 aa
ALB: 1133 aa
INS: 110 aa
HBB: 147 aa
HBA1: 142 aa
LDHA: 332 aa
PKM: 531 aa
ENO1: 434 aa
ALDOA: 364 aa
TPI1: 249 aa
PGK1: 417 aa
CAT: 527 aa
SOD1: 154 aa
SOD2: 222 aa
GPX1: 203 aa
MB: 154 aa
TTN: 34350 aa
COL1A1: 1464 aa
FN1: 2477 a

In [3]:
from Bio.SeqUtils.ProtParam import ProteinAnalysis

AA_LIST = list("ACDEFGHIKLMNPQRSTVWY")

def extract_features(seq):
    seq = seq.replace("X", "").replace("U", "").replace("B", "").replace("Z", "")
    if len(seq) < 10:
        return None
    analysis = ProteinAnalysis(seq)
    aa_comp = analysis.get_amino_acids_percent()

    basic = aa_comp.get("K", 0) + aa_comp.get("R", 0) + aa_comp.get("H", 0)
    acidic = aa_comp.get("D", 0) + aa_comp.get("E", 0)

    features = {f"frac_{aa}": aa_comp.get(aa, 0) for aa in AA_LIST}
    features["length"] = len(seq)
    features["basic_fraction"] = basic
    features["acidic_fraction"] = acidic
    features["net_charge_proxy"] = basic - acidic
    features["isoelectric_point"] = analysis.isoelectric_point()
    features["instability_index"] = analysis.instability_index()
    features["aromaticity"] = analysis.aromaticity()
    features["gravy"] = analysis.gravy()  # hydrophobicity

    return features

feature_rows = []
for _, row in df.iterrows():
    feats = extract_features(row["sequence"])
    if feats:
        feats["gene"] = row["gene"]
        feats["is_tf"] = row["is_tf"]
        feature_rows.append(feats)

feat_df = pd.DataFrame(feature_rows)
print(f"Feature matrix shape: {feat_df.shape}")
print(f"\nMean basic_fraction - TFs: {feat_df[feat_df.is_tf==1]['basic_fraction'].mean():.4f}")
print(f"Mean basic_fraction - non-TFs: {feat_df[feat_df.is_tf==0]['basic_fraction'].mean():.4f}")
print(f"\nMean isoelectric_point - TFs: {feat_df[feat_df.is_tf==1]['isoelectric_point'].mean():.2f}")
print(f"Mean isoelectric_point - non-TFs: {feat_df[feat_df.is_tf==0]['isoelectric_point'].mean():.2f}")

AttributeError: 'ProteinAnalysis' object has no attribute 'get_amino_acids_percent'

In [4]:
seq_test = df.iloc[0]["sequence"]
analysis_test = ProteinAnalysis(seq_test)
print([m for m in dir(analysis_test) if not m.startswith("_")])

['amino_acids_content', 'amino_acids_percent', 'aromaticity', 'charge_at_pH', 'count_amino_acids', 'flexibility', 'gravy', 'instability_index', 'isoelectric_point', 'length', 'molar_extinction_coefficient', 'molecular_weight', 'monoisotopic', 'protein_scale', 'secondary_structure_fraction', 'sequence']


In [5]:
def extract_features(seq):
    seq = seq.replace("X", "").replace("U", "").replace("B", "").replace("Z", "")
    if len(seq) < 10:
        return None
    analysis = ProteinAnalysis(seq)
    aa_comp = analysis.amino_acids_percent

    basic = aa_comp.get("K", 0) + aa_comp.get("R", 0) + aa_comp.get("H", 0)
    acidic = aa_comp.get("D", 0) + aa_comp.get("E", 0)

    features = {f"frac_{aa}": aa_comp.get(aa, 0) for aa in AA_LIST}
    features["length"] = len(seq)
    features["basic_fraction"] = basic
    features["acidic_fraction"] = acidic
    features["net_charge_proxy"] = basic - acidic
    features["isoelectric_point"] = analysis.isoelectric_point()
    features["instability_index"] = analysis.instability_index()
    features["aromaticity"] = analysis.aromaticity()
    features["gravy"] = analysis.gravy()

    return features

feature_rows = []
for _, row in df.iterrows():
    feats = extract_features(row["sequence"])
    if feats:
        feats["gene"] = row["gene"]
        feats["is_tf"] = row["is_tf"]
        feature_rows.append(feats)

feat_df = pd.DataFrame(feature_rows)
print(f"Feature matrix shape: {feat_df.shape}")
print(f"\nMean basic_fraction - TFs: {feat_df[feat_df.is_tf==1]['basic_fraction'].mean():.4f}")
print(f"Mean basic_fraction - non-TFs: {feat_df[feat_df.is_tf==0]['basic_fraction'].mean():.4f}")
print(f"\nMean isoelectric_point - TFs: {feat_df[feat_df.is_tf==1]['isoelectric_point'].mean():.2f}")
print(f"Mean isoelectric_point - non-TFs: {feat_df[feat_df.is_tf==0]['isoelectric_point'].mean():.2f}")

Feature matrix shape: (100, 30)

Mean basic_fraction - TFs: 13.0237
Mean basic_fraction - non-TFs: 14.9057

Mean isoelectric_point - TFs: 7.10
Mean isoelectric_point - non-TFs: 7.10


In [6]:
print(feat_df[["gene", "is_tf", "isoelectric_point", "basic_fraction", "length"]].sort_values("is_tf").to_string())

       gene  is_tf  isoelectric_point  basic_fraction  length
60     TPI1      0           6.445849       12.851406     249
61     PGK1      0           8.302319       13.908873     417
51    GAPDH      0           8.566124       13.731343     335
50     ACTB      0           5.290483       12.266667     375
54      HBB      0           6.744538       15.646259     147
55     HBA1      0           8.716915       16.901408     142
53      INS      0           5.218581        8.181818     110
52      ALB      0           6.618810       16.946161    1133
62      CAT      0           6.897605       15.370019     527
63     SOD1      0           5.696086       14.935065     154
58     ENO1      0           7.011794       14.055300     434
59    ALDOA      0           8.299611       13.736264     364
57      PKM      0           7.955028       15.065913     531
56     LDHA      0           8.443312       13.855422     332
81  ATP5F1A      0           9.157363       13.562387     553
80     C

In [7]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

feature_cols = [c for c in feat_df.columns if c not in ["gene", "is_tf"]]
X = feat_df[feature_cols].values
y = feat_df["is_tf"].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

rf = RandomForestClassifier(n_estimators=200, max_depth=5, random_state=42)
rf_scores = cross_val_score(rf, X, y, cv=cv, scoring="accuracy")

lr = LogisticRegression(max_iter=1000)
lr_scores = cross_val_score(lr, X_scaled, y, cv=cv, scoring="accuracy")

print(f"Random Forest 5-fold CV accuracy: {rf_scores.mean():.3f} (+/- {rf_scores.std():.3f})")
print(f"Individual folds: {rf_scores}")
print(f"\nLogistic Regression 5-fold CV accuracy: {lr_scores.mean():.3f} (+/- {lr_scores.std():.3f})")
print(f"Individual folds: {lr_scores}")

# Feature importance from Random Forest (fit on full data for interpretability)
rf.fit(X, y)
importances = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)
print(f"\nTop 10 most important features:")
print(importances.head(10))

Random Forest 5-fold CV accuracy: 0.880 (+/- 0.040)
Individual folds: [0.9  0.85 0.95 0.85 0.85]

Logistic Regression 5-fold CV accuracy: 0.870 (+/- 0.051)
Individual folds: [0.85 0.9  0.95 0.8  0.85]

Top 10 most important features:
instability_index    0.187202
frac_P               0.149151
frac_S               0.134299
frac_V               0.087700
gravy                0.042991
frac_I               0.034082
basic_fraction       0.030644
frac_C               0.027550
frac_R               0.024039
frac_K               0.024023
dtype: float64


In [8]:
print("Instability index comparison:")
print(f"TFs mean: {feat_df[feat_df.is_tf==1]['instability_index'].mean():.2f}")
print(f"Non-TFs mean: {feat_df[feat_df.is_tf==0]['instability_index'].mean():.2f}")

print("\nProline fraction comparison:")
print(f"TFs mean: {feat_df[feat_df.is_tf==1]['frac_P'].mean():.4f}")
print(f"Non-TFs mean: {feat_df[feat_df.is_tf==0]['frac_P'].mean():.4f}")

print("\nSerine fraction comparison:")
print(f"TFs mean: {feat_df[feat_df.is_tf==1]['frac_S'].mean():.4f}")
print(f"Non-TFs mean: {feat_df[feat_df.is_tf==0]['frac_S'].mean():.4f}")

from scipy import stats
for feat in ["instability_index", "frac_P", "frac_S", "basic_fraction"]:
    t, p = stats.ttest_ind(feat_df[feat_df.is_tf==1][feat], feat_df[feat_df.is_tf==0][feat])
    print(f"\n{feat}: t={t:.3f}, p={p:.4f}")

Instability index comparison:
TFs mean: 59.35
Non-TFs mean: 36.76

Proline fraction comparison:
TFs mean: 8.7973
Non-TFs mean: 4.8850

Serine fraction comparison:
TFs mean: 10.1874
Non-TFs mean: 6.3027

instability_index: t=9.522, p=0.0000

frac_P: t=7.368, p=0.0000

frac_S: t=8.556, p=0.0000

basic_fraction: t=-3.368, p=0.0011


In [9]:
# 1. TTN's effect on length stats
print("=== Length stats WITH vs WITHOUT TTN ===")
print(f"With TTN    - mean: {feat_df['length'].mean():.1f}, median: {feat_df['length'].median():.1f}")
no_ttn = feat_df[feat_df['gene'] != 'TTN']
print(f"Without TTN - mean: {no_ttn['length'].mean():.1f}, median: {no_ttn['length'].median():.1f}")

# 2. RPS6's effect on basic_fraction comparison
print("\n=== basic_fraction: WITH vs WITHOUT RPS6 in non-TF group ===")
non_tf_with = feat_df[feat_df.is_tf==0]['basic_fraction']
non_tf_without = feat_df[(feat_df.is_tf==0) & (feat_df.gene != 'RPS6')]['basic_fraction']
print(f"Non-TF mean WITH RPS6:    {non_tf_with.mean():.4f}")
print(f"Non-TF mean WITHOUT RPS6: {non_tf_without.mean():.4f}")
print(f"TF mean (unchanged):      {feat_df[feat_df.is_tf==1]['basic_fraction'].mean():.4f}")

t_with, p_with = stats.ttest_ind(feat_df[feat_df.is_tf==1]['basic_fraction'], non_tf_with)
t_without, p_without = stats.ttest_ind(feat_df[feat_df.is_tf==1]['basic_fraction'], non_tf_without)
print(f"\nt-test WITH RPS6:    t={t_with:.3f}, p={p_with:.4f}")
print(f"t-test WITHOUT RPS6: t={t_without:.3f}, p={p_without:.4f}")

# 3. Homology risk - check paralog families and test leave-family-out
paralog_families = {
    "GATA": ["GATA1","GATA2","GATA3","GATA4"],
    "FOX":  ["FOXP3","FOXA2","FOXO1"],
    "STAT": ["STAT3","STAT1"],
}
print("\n=== Paralog families present in TF set ===")
for fam, members in paralog_families.items():
    present = [m for m in members if m in feat_df['gene'].values]
    print(f"{fam}: {present}")

# Leave-one-family-out test: train excluding GATA family, test ONLY on GATA family
gata_mask = feat_df['gene'].isin(paralog_families["GATA"])
train_df = feat_df[~gata_mask]
test_df = feat_df[gata_mask]

X_train = train_df[feature_cols].values
y_train = train_df['is_tf'].values
X_test = test_df[feature_cols].values
y_test = test_df['is_tf'].values

rf_loo = RandomForestClassifier(n_estimators=200, max_depth=5, random_state=42)
rf_loo.fit(X_train, y_train)
acc = rf_loo.score(X_test, y_test)
print(f"\nLeave-GATA-family-out: trained on {len(train_df)} proteins (no GATA), tested on {len(test_df)} GATA proteins")
print(f"Accuracy on held-out GATA family: {acc:.3f}")
print(f"Predictions: {rf_loo.predict(X_test)}, True labels: {y_test}")

=== Length stats WITH vs WITHOUT TTN ===
With TTN    - mean: 843.2, median: 430.5
Without TTN - mean: 504.7, median: 430.0

=== basic_fraction: WITH vs WITHOUT RPS6 in non-TF group ===
Non-TF mean WITH RPS6:    14.9057
Non-TF mean WITHOUT RPS6: 14.6280
TF mean (unchanged):      13.0237

t-test WITH RPS6:    t=-3.368, p=0.0011
t-test WITHOUT RPS6: t=-3.275, p=0.0015

=== Paralog families present in TF set ===
GATA: ['GATA1', 'GATA2', 'GATA3', 'GATA4']
FOX: ['FOXP3', 'FOXA2', 'FOXO1']
STAT: ['STAT3', 'STAT1']

Leave-GATA-family-out: trained on 96 proteins (no GATA), tested on 4 GATA proteins
Accuracy on held-out GATA family: 1.000
Predictions: [1 1 1 1], True labels: [1 1 1 1]


In [10]:
!pip install git+https://github.com/doszilab/AIUPred.git -q

from aiupred import AIUPred
import numpy as np

predictor = AIUPred()

# SOX2 sequence - already have it from Step 2 fetching (df has gene/sequence columns)
sox2_seq = df[df["gene"] == "SOX2"]["sequence"].values[0]
disorder_scores = predictor.predict_disorder(sox2_seq)

print(f"SOX2 length: {len(sox2_seq)}")
print(f"Disorder score array length: {len(disorder_scores)}")
print(f"Mean disorder: {np.mean(disorder_scores):.3f}")
print(f"Fraction of residues with disorder >= 0.5: {np.mean(np.array(disorder_scores) >= 0.5):.3f}")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
SOX2 length: 317
Disorder score array length: 317
Mean disorder: 0.887
Fraction of residues with disorder >= 0.5: 0.905


In [11]:
def max_consecutive_disorder(scores, threshold=0.5):
    is_disordered = np.array(scores) >= threshold
    max_run = current_run = 0
    for val in is_disordered:
        if val:
            current_run += 1
            max_run = max(max_run, current_run)
        else:
            current_run = 0
    return max_run

disorder_rows = []
for _, row in df.iterrows():
    seq = row["sequence"].replace("X", "").replace("U", "").replace("B", "").replace("Z", "")
    if len(seq) < 10:
        continue
    scores = predictor.predict_disorder(seq)
    scores = np.array(scores)
    disorder_rows.append({
        "gene": row["gene"],
        "is_tf": row["is_tf"],
        "mean_disorder": float(np.mean(scores)),
        "median_disorder": float(np.median(scores)),
        "idr_fraction": float(np.mean(scores >= 0.5)),
        "max_disorder_stretch": max_consecutive_disorder(scores)
    })
    print(f"{row['gene']}: mean_disorder={np.mean(scores):.3f}, idr_fraction={np.mean(scores>=0.5):.3f}")

disorder_df = pd.DataFrame(disorder_rows)
print(f"\nProcessed {len(disorder_df)} proteins")

print("\n=== Group comparison ===")
for feat in ["mean_disorder", "median_disorder", "idr_fraction", "max_disorder_stretch"]:
    tf_mean = disorder_df[disorder_df.is_tf==1][feat].mean()
    non_tf_mean = disorder_df[disorder_df.is_tf==0][feat].mean()
    t, p = stats.ttest_ind(disorder_df[disorder_df.is_tf==1][feat], disorder_df[disorder_df.is_tf==0][feat])
    print(f"{feat}: TF={tf_mean:.3f}, non-TF={non_tf_mean:.3f}, t={t:.3f}, p={p:.4f}")

TP53: mean_disorder=0.559, idr_fraction=0.527
MYC: mean_disorder=0.733, idr_fraction=0.837
JUN: mean_disorder=0.706, idr_fraction=0.822
FOS: mean_disorder=0.881, idr_fraction=0.966
GATA1: mean_disorder=0.726, idr_fraction=0.724
GATA2: mean_disorder=0.825, idr_fraction=0.829
GATA3: mean_disorder=0.841, idr_fraction=0.871
GATA4: mean_disorder=0.833, idr_fraction=0.817
SOX2: mean_disorder=0.887, idr_fraction=0.905
POU5F1: mean_disorder=0.640, idr_fraction=0.597
KLF4: mean_disorder=0.834, idr_fraction=0.844
NANOG: mean_disorder=0.736, idr_fraction=0.698
STAT3: mean_disorder=0.238, idr_fraction=0.142
NFKB1: mean_disorder=0.302, idr_fraction=0.246
RELA: mean_disorder=0.592, idr_fraction=0.581
CREB1: mean_disorder=0.696, idr_fraction=0.786
E2F1: mean_disorder=0.619, idr_fraction=0.611
FOXP3: mean_disorder=0.540, idr_fraction=0.510
PAX6: mean_disorder=0.642, idr_fraction=0.585
HOXA1: mean_disorder=0.766, idr_fraction=0.761
IRF1: mean_disorder=0.734, idr_fraction=0.766
SMAD3: mean_disorder=0.35

In [12]:
merged_df = feat_df.merge(disorder_df[["gene", "mean_disorder", "median_disorder", "idr_fraction", "max_disorder_stretch"]], on="gene")

idr_feature_cols = feature_cols + ["mean_disorder", "median_disorder", "idr_fraction", "max_disorder_stretch"]
X_idr = merged_df[idr_feature_cols].values
y_idr = merged_df["is_tf"].values

rf_idr = RandomForestClassifier(n_estimators=200, max_depth=5, random_state=42)
rf_idr_scores = cross_val_score(rf_idr, X_idr, y_idr, cv=cv, scoring="accuracy")

# disorder features ALONE, no proxies
X_disorder_only = merged_df[["mean_disorder", "median_disorder", "idr_fraction", "max_disorder_stretch"]].values
rf_disorder_only = RandomForestClassifier(n_estimators=200, max_depth=5, random_state=42)
rf_disorder_only_scores = cross_val_score(rf_disorder_only, X_disorder_only, y_idr, cv=cv, scoring="accuracy")

print(f"Original proxy-feature model:        {rf_scores.mean():.3f} (+/- {rf_scores.std():.3f})")
print(f"Proxy + real IDR features combined:  {rf_idr_scores.mean():.3f} (+/- {rf_idr_scores.std():.3f})")
print(f"Real IDR features ALONE:              {rf_disorder_only_scores.mean():.3f} (+/- {rf_disorder_only_scores.std():.3f})")

rf_idr.fit(X_idr, y_idr)
importances_idr = pd.Series(rf_idr.feature_importances_, index=idr_feature_cols).sort_values(ascending=False)
print(f"\nTop 10 features with real IDR data included:")
print(importances_idr.head(10))

Original proxy-feature model:        0.880 (+/- 0.040)
Proxy + real IDR features combined:  0.870 (+/- 0.051)
Real IDR features ALONE:              0.870 (+/- 0.040)

Top 10 features with real IDR data included:
instability_index       0.134389
max_disorder_stretch    0.101564
median_disorder         0.097430
frac_P                  0.092039
frac_S                  0.084865
mean_disorder           0.071699
idr_fraction            0.049982
frac_V                  0.047026
frac_C                  0.030712
length                  0.027739
dtype: float64
